# Final discourse analysis (RQ2) — canonical thesis results

**This notebook generates every final discourse table and figure in the thesis.**

| notebook | role |
|---|---|
| `2_discopy_discourse_analysis.ipynb` | parser diagnostics and inspection (DiMLex alignment, coverage gap, NoSense retention) |
| `3`–`6` | manual validation, all complete and closed |
| **`7_final_discourse_analysis.ipynb`** | **canonical thesis-result generation** |

## Scope

The pipeline is frozen: **native standard discopy, explicit relations only**.
The single input is `discopy_explicit_candidates.csv` filtered to
`is_connective == True`. Nothing here reads DiMLex, the forced-span probe or the
rejected hybrid.

Produced:

* **A** overall explicit-relation density
* **B** top-level PDTB relation density (Comparison, Contingency, Expansion, Temporal)
* **C** relative four-class composition
* **D** level-2 PDTB senses (secondary descriptive)
* **E** paired game-level bootstrap of pairwise model differences

Deliberately excluded: discourse-only co-occurrence (deferred to the joint
discourse × semantic analysis), DiMLex lexical counts as results, hybrid or
forced-span relations, semantic categories, correctness stratification, and
complexity indices.

## Aggregation rules

* descriptive statistics are computed **per run**, then reported as mean ± SD
  across the **three stochastic runs**;
* **greedy is a single run**, always kept separate, SD undefined (NaN);
* the bootstrap resamples **games**, not justifications, with the **same**
  resampled game ids for every model.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_repo_root(start=None, repo_name="masters_thesis_sdg"):
    current = (start or Path.cwd()).resolve()
    while current.name != repo_name:
        if current.parent == current:
            raise FileNotFoundError(f"repo root {repo_name!r} not found")
        current = current.parent
    return current


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.justification_analysis.comparison import discourse_final as fin
from src.justification_analysis.comparison import discourse_figures as figs
from src.justification_analysis.pipeline import config as pipeline_config
from src.justification_analysis.pipeline import corpus as corpus_module
from src.justification_analysis.pipeline import manifest as manifest_module

# ---------------------------------------------------------------------------
# THE ONE THING TO CHANGE when moving to fine-tuned outputs.
# Everything below - inputs, artifact freshness, and every output path -
# derives from this. There is no other path to edit, and no fallback to base.
# ---------------------------------------------------------------------------
CONFIG = pipeline_config.default_config(stage="base", repo_root=REPO_ROOT)

FINAL_TABLES = CONFIG.final_discourse_tables
FINAL_FIGURES = CONFIG.final_discourse_figures
FINAL_TABLES.mkdir(parents=True, exist_ok=True)
FINAL_FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

print("tables :", FINAL_TABLES.relative_to(REPO_ROOT))
print("figures:", FINAL_FIGURES.relative_to(REPO_ROOT))

tables : analysis\cross_model\base\voting\prompt_v4\justification_analysis\discourse_parser\thesis_tables\final_discourse
figures: analysis\cross_model\base\voting\prompt_v4\justification_analysis\discourse_parser\figures\final_discourse


## 1. Input

The corpus and the parser artifact both come from `CONFIG`. The candidate
table passes the **manifest freshness gate**: if its corpus fingerprint does
not match the corpus loaded here, this cell raises rather than quietly
analysing a stale artifact. The frozen base counts are checked only when the
active stage is `base`; structural invariants are checked for every stage.

In [2]:
accepted, justifications = fin.load_production_data(CONFIG)

# Provenance, printed before any number: which corpus this is, which artifact
# was consumed, and the parser metadata read FROM THE MANIFEST rather than
# retyped into markdown where it can go stale.
print(manifest_module.provenance_block(
    CONFIG, justifications, accepted.attrs.get("manifest")))

print()
print(f"accepted explicit relations : {len(accepted):,}")
print(f"justifications              : {len(justifications):,}")
print(f"games                       : {justifications['game_id'].nunique()}")
print(f"words (WORD_PATTERN)        : {justifications['n_words'].sum():,}")
print(f"level-2 senses observed     : {accepted['raw_sense'].nunique()}")
print("\nrelations by top-level class:")
display(accepted["top_level"].value_counts().rename("n").to_frame())

  STAGE            : base
  PROMPT VERSION   : prompt_v4
  MODELS           : Gemma 4 2B, Gemma 4 4B, Gemma 4 31B
  RUNS             : stochastic run_1, run_2, run_3 | greedy greedy_t0
  JUSTIFICATIONS   : 2,292
  GAMES            : 191
  SENTENCES        : 8,044
  WORDS            : 169,748
  CORPUS HASH      : a0946232dd495b38779fe78a4788175d9a240f686b6df9f56a701b6f9f5614e1
  ARTIFACT DIR     : analysis\cross_model\base\voting\prompt_v4\justification_analysis
------------------------------------------------------------------------
  ARTIFACT         : discopy_explicit_candidates.csv
  BUILT            : 2026-08-27T22:17:03.212319+00:00
  PARSER           : rknaebel/discopy 1.1.0
  CHECKPOINT       : bert-10.11.21-13.31
  BACKBONE         : bert-base-cased
  FRESHNESS        : verified against the current corpus hash

accepted explicit relations : 5,504
justifications              : 2,292
games                       : 191
words (WORD_PATTERN)        : 169,748
level-2 senses observed  

,n
top_level,
Comparison,1608
Contingency,1534
Expansion,1513
Temporal,849


## 2. F0 — per-run base table

Every table below aggregates this one, so the un-aggregated numbers stay
inspectable and no statistic is computed twice by two different routes.

In [3]:
run_level = fin.run_level_table(accepted, justifications)
display(run_level.round(3))

,model,decoding_group,run_label,n_justifications,total_words,total_relations,n_justifications_with_relation,n_Comparison,n_Contingency,n_Expansion,n_Temporal,relations_per_100_words,relations_per_justification,pct_justifications_with_relation,Comparison_per_100_words,Comparison_pct_of_relations,Contingency_per_100_words,Contingency_pct_of_relations,Expansion_per_100_words,Expansion_pct_of_relations,Temporal_per_100_words,Temporal_pct_of_relations
0,Gemma 4 2B,Stochastic,run_1,191,12409,419,188,116,130,131,42,3.377,2.194,98.429,0.935,27.685,1.048,31.026,1.056,31.265,0.338,10.024
1,Gemma 4 2B,Stochastic,run_2,191,12474,421,183,121,141,127,32,3.375,2.204,95.812,0.970,28.741,1.130,33.492,1.018,30.166,0.257,7.601
2,Gemma 4 2B,Stochastic,run_3,191,12377,396,186,113,153,102,28,3.199,2.073,97.382,0.913,28.535,1.236,38.636,0.824,25.758,0.226,7.071
3,Gemma 4 2B,Greedy,greedy_t0,191,12338,441,186,125,167,124,25,3.574,2.309,97.382,1.013,28.345,1.354,37.868,1.005,28.118,0.203,5.669
4,Gemma 4 4B,Stochastic,run_1,191,15655,437,183,185,78,93,81,2.791,2.288,95.812,1.182,42.334,0.498,17.849,0.594,21.281,0.517,18.535
5,Gemma 4 4B,Stochastic,run_2,191,15601,456,185,188,91,103,74,2.923,2.387,96.859,1.205,41.228,0.583,19.956,0.660,22.588,0.474,16.228
6,Gemma 4 4B,Stochastic,run_3,191,15696,422,181,191,70,110,51,2.689,2.209,94.764,1.217,45.261,0.446,16.588,0.701,26.066,0.325,12.085
7,Gemma 4 4B,Greedy,greedy_t0,191,15547,456,188,175,99,111,71,2.933,2.387,98.429,1.126,38.377,0.637,21.711,0.714,24.342,0.457,15.570
8,Gemma 4 31B,Stochastic,run_1,191,14445,537,183,105,159,148,125,3.718,2.812,95.812,0.727,19.553,1.101,29.609,1.025,27.561,0.865,23.277
9,Gemma 4 31B,Stochastic,run_2,191,14342,489,183,102,144,144,99,3.410,2.560,95.812,0.711,20.859,1.004,29.448,1.004,29.448,0.690,20.245


## 3. A — overall explicit-relation density (F1)

In [4]:
f1 = fin.overall_density(run_level)
display(f1.round(3))

,model,decoding_group,n_runs,relations_per_100_words_mean,relations_per_100_words_sd,relations_per_justification_mean,relations_per_justification_sd,pct_justifications_with_relation_mean,pct_justifications_with_relation_sd
0,Gemma 4 2B,Stochastic,3,3.317,0.102,2.157,0.073,97.208,1.318
1,Gemma 4 2B,Greedy,1,3.574,NaN,2.309,NaN,97.382,NaN
2,Gemma 4 4B,Stochastic,3,2.801,0.117,2.295,0.089,95.812,1.047
3,Gemma 4 4B,Greedy,1,2.933,NaN,2.387,NaN,98.429,NaN
4,Gemma 4 31B,Stochastic,3,3.542,0.159,2.677,0.127,95.812,0.000
5,Gemma 4 31B,Greedy,1,3.640,NaN,2.733,NaN,94.764,NaN


## 4. B — top-level PDTB relation density (F2)

The primary cross-model comparison.

In [5]:
f2 = fin.top_level_density(run_level)
display(f2.round(3))

,model,decoding_group,n_runs,Contingency_per_100_words_mean,Contingency_per_100_words_sd,Comparison_per_100_words_mean,Comparison_per_100_words_sd,Expansion_per_100_words_mean,Expansion_per_100_words_sd,Temporal_per_100_words_mean,Temporal_per_100_words_sd
0,Gemma 4 2B,Stochastic,3,1.138,0.095,0.939,0.029,0.966,0.124,0.274,0.058
1,Gemma 4 2B,Greedy,1,1.354,NaN,1.013,NaN,1.005,NaN,0.203,NaN
2,Gemma 4 4B,Stochastic,3,0.509,0.069,1.201,0.018,0.652,0.054,0.439,0.101
3,Gemma 4 4B,Greedy,1,0.637,NaN,1.126,NaN,0.714,NaN,0.457,NaN
4,Gemma 4 31B,Stochastic,3,1.067,0.054,0.688,0.054,1.023,0.018,0.764,0.091
5,Gemma 4 31B,Greedy,1,0.997,NaN,0.669,NaN,1.178,NaN,0.795,NaN


## 5. C — relative four-class composition (F3)

Shares are computed within each run and then averaged, so a run that happened
to produce more relations does not dominate the mean. This differs slightly
from a pooled share.

In [6]:
f3 = fin.top_level_composition(run_level)
display(f3.round(2))

,model,decoding_group,n_runs,Contingency_pct_of_relations_mean,Contingency_pct_of_relations_sd,Comparison_pct_of_relations_mean,Comparison_pct_of_relations_sd,Expansion_pct_of_relations_mean,Expansion_pct_of_relations_sd,Temporal_pct_of_relations_mean,Temporal_pct_of_relations_sd
0,Gemma 4 2B,Stochastic,3,34.38,3.88,28.32,0.56,29.06,2.91,8.23,1.57
1,Gemma 4 2B,Greedy,1,37.87,NaN,28.34,NaN,28.12,NaN,5.67,NaN
2,Gemma 4 4B,Stochastic,3,18.13,1.70,42.94,2.08,23.31,2.47,15.62,3.27
3,Gemma 4 4B,Greedy,1,21.71,NaN,38.38,NaN,24.34,NaN,15.57,NaN
4,Gemma 4 31B,Stochastic,3,30.12,1.03,19.44,1.48,28.91,1.18,21.53,1.57
5,Gemma 4 31B,Greedy,1,27.39,NaN,18.39,NaN,32.38,NaN,21.84,NaN


## 6. D — level-2 PDTB senses (F4), secondary

Only senses the corpus actually contains are reported: the checkpoint supports
16 non-`NoSense` labels, the corpus uses 9. `Expansion.Restatement` has **n = 1**
corpus-wide and is kept in the table — dropping it would change the share
denominator.

In [7]:
f4 = fin.fine_grained_senses(accepted, justifications)
display(f4.round(3))

print("corpus totals by sense:")
display(
    f4.groupby(["top_level", "raw_sense"], observed=True)["total_count"]
    .sum().sort_values(ascending=False).to_frame("n")
)

,model,decoding_group,top_level,raw_sense,n_runs,total_count,mean_count_per_run,mean_per_100_words,sd_per_100_words,mean_pct_of_relations,sd_pct_of_relations
0,Gemma 4 2B,Stochastic,Comparison,Comparison.Concession,3,8,2.667,0.021,0.009,0.654,0.308
1,Gemma 4 2B,Stochastic,Comparison,Comparison.Contrast,3,342,114.000,0.918,0.037,27.666,0.543
2,Gemma 4 2B,Stochastic,Contingency,Contingency.Cause,3,357,119.000,0.958,0.091,28.951,3.524
3,Gemma 4 2B,Stochastic,Contingency,Contingency.Condition,3,67,22.333,0.180,0.017,5.433,0.657
4,Gemma 4 2B,Stochastic,Expansion,Expansion.Alternative,3,13,4.333,0.035,0.017,1.041,0.479
5,Gemma 4 2B,Stochastic,Expansion,Expansion.Conjunction,3,347,115.667,0.931,0.109,28.022,2.489
6,Gemma 4 2B,Stochastic,Expansion,Expansion.Restatement,3,0,0.000,0.000,0.000,0.000,0.000
7,Gemma 4 2B,Stochastic,Temporal,Temporal.Asynchronous,3,69,23.000,0.185,0.032,5.568,0.828
8,Gemma 4 2B,Stochastic,Temporal,Temporal.Synchrony,3,33,11.000,0.089,0.028,2.663,0.797
9,Gemma 4 2B,Greedy,Comparison,Comparison.Concession,1,2,2.000,0.016,NaN,0.454,NaN


corpus totals by sense:


,,n
top_level,raw_sense,
Comparison,Comparison.Contrast,1576
Expansion,Expansion.Conjunction,1473
Contingency,Contingency.Cause,1103
Temporal,Temporal.Asynchronous,666
Contingency,Contingency.Condition,431
Temporal,Temporal.Synchrony,183
Expansion,Expansion.Alternative,39
Comparison,Comparison.Concession,32
Expansion,Expansion.Restatement,1


## 7. E — paired game-level bootstrap (F5)

One replicate:

1. resample the 191 game ids with replacement;
2. use the **same** resampled ids for every model;
3. keep all three stochastic realisations of each sampled game;
4. compute the density separately for each run;
5. average the three run values into one model value;
6. difference the models.

Stochastic and greedy are bootstrapped separately. Greedy has a single run, so
its interval reflects between-game variation only. 95% percentile intervals,
fixed seed, no p-values.

In [8]:
f5 = fin.paired_game_bootstrap(
    accepted, justifications,
    n_replicates=fin.BOOTSTRAP_REPLICATES,
    seed=fin.BOOTSTRAP_SEED,
)
print(f"{fin.BOOTSTRAP_REPLICATES:,} replicates, seed {fin.BOOTSTRAP_SEED}")

for decoding in ("Stochastic", "Greedy"):
    print(f"\n{decoding}")
    display(
        f5.loc[f5["decoding_group"].astype(str).eq(decoding),
               ["metric", "model_a", "model_b", "difference",
                "ci_low", "ci_high", "ci_excludes_zero"]]
        .round(3).reset_index(drop=True)
    )

10,000 replicates, seed 20260826

Stochastic


,metric,model_a,model_b,difference,ci_low,ci_high,ci_excludes_zero
0,All relations,Gemma 4 2B,Gemma 4 31B,-0.225,-0.453,-0.002,True
1,All relations,Gemma 4 2B,Gemma 4 4B,0.516,0.328,0.708,True
2,All relations,Gemma 4 4B,Gemma 4 31B,-0.741,-0.958,-0.523,True
3,Contingency,Gemma 4 2B,Gemma 4 31B,0.071,-0.062,0.203,False
4,Contingency,Gemma 4 2B,Gemma 4 4B,0.629,0.513,0.741,True
5,Contingency,Gemma 4 4B,Gemma 4 31B,-0.557,-0.678,-0.439,True
6,Comparison,Gemma 4 2B,Gemma 4 31B,0.251,0.141,0.360,True
7,Comparison,Gemma 4 2B,Gemma 4 4B,-0.262,-0.355,-0.166,True
8,Comparison,Gemma 4 4B,Gemma 4 31B,0.513,0.407,0.618,True
9,Expansion,Gemma 4 2B,Gemma 4 31B,-0.057,-0.211,0.095,False



Greedy


,metric,model_a,model_b,difference,ci_low,ci_high,ci_excludes_zero
0,All relations,Gemma 4 2B,Gemma 4 31B,-0.065,-0.442,0.330,False
1,All relations,Gemma 4 2B,Gemma 4 4B,0.641,0.331,0.955,True
2,All relations,Gemma 4 4B,Gemma 4 31B,-0.707,-1.030,-0.378,True
3,Contingency,Gemma 4 2B,Gemma 4 31B,0.356,0.141,0.576,True
4,Contingency,Gemma 4 2B,Gemma 4 4B,0.717,0.512,0.916,True
5,Contingency,Gemma 4 4B,Gemma 4 31B,-0.360,-0.550,-0.169,True
6,Comparison,Gemma 4 2B,Gemma 4 31B,0.344,0.187,0.503,True
7,Comparison,Gemma 4 2B,Gemma 4 4B,-0.112,-0.276,0.052,False
8,Comparison,Gemma 4 4B,Gemma 4 31B,0.456,0.290,0.618,True
9,Expansion,Gemma 4 2B,Gemma 4 31B,-0.173,-0.401,0.051,False


## 8. Write the final tables

In [9]:
tables = fin.build_final_tables(accepted, justifications)
for path in fin.write_final_tables(tables, FINAL_TABLES):
    print("wrote", path.name)

wrote F0_run_level.csv
wrote F1_overall_density.csv
wrote F2_top_level_density.csv
wrote F3_top_level_composition.csv
wrote F4_fine_grained_senses.csv
wrote F5_bootstrap_pairwise.csv


## 9. Figures

Three figures, deliberately. No discourse-only co-occurrence figure.

In [10]:
for path in figs.build_final_figures(tables, FINAL_FIGURES):
    print("wrote", path.name)

wrote F1_top_level_density.png
wrote F2_composition.png
wrote F3_fine_grained_senses.png
wrote F4_bootstrap_forest_stochastic.png
wrote F5_bootstrap_forest_greedy.png


## 10. Method evidence — diagnostic, **not** a result

One figure and one table supporting the Method's *Contextual Relation
Identification* paragraph: how often each candidate form survives the
contextual `NoSense` filter. These describe the parser, not the models, and are
written outside the final-results directories.

In [11]:
from src.justification_analysis.comparison import discourse_comparison as dc
from src.justification_analysis.comparison import discourse_statistics as ds

# Through the freshness gate, like everything else - not read by path.
candidates, _manifest = manifest_module.load_verified_candidates(
    CONFIG, justifications)
candidates = dc.normalise_candidates(candidates)
forms = ds.connective_form_statistics(candidates, justifications)

n_candidates = len(candidates)
n_accepted = int(candidates["is_connective"].sum())
print(f"{n_candidates:,} candidates -> {n_accepted:,} accepted "
      f"({100 * (n_candidates - n_accepted) / n_candidates:.1f}% rejected as NoSense)")
display(forms["acceptance_by_form"].head(15))

diagnostic_tables = CONFIG.discourse_dir / "thesis_tables"
diagnostic_figures = CONFIG.discourse_dir / "figures" / "method"
diagnostic_tables.mkdir(parents=True, exist_ok=True)
diagnostic_figures.mkdir(parents=True, exist_ok=True)
acceptance = forms["acceptance_by_form"].set_index("form")
acceptance.to_csv(diagnostic_tables / "02c_connective_form_acceptance.csv",
                  encoding="utf-8-sig")
ds.to_latex(acceptance, diagnostic_tables / "02c_connective_form_acceptance.tex",
            caption="Candidate acceptance rate per connective form")
figs.figure_contextual_filtering(
    forms["acceptance_by_form"],
    diagnostic_figures / "fig4_contextual_filtering.png",
)
print("wrote the method diagnostic table and figure")

14,209 candidates -> 5,504 accepted (61.3% rejected as NoSense)


,form,n_candidates,n_accepted,n_rejected_nosense,pct_accepted
0,and,5600,1035,4565,18.48
1,while,891,890,1,99.89
2,since,676,676,0,100.00
3,if,486,426,60,87.65
4,however,291,291,0,100.00
5,later,290,285,5,98.28
6,although,242,242,0,100.00
7,also,245,237,8,96.73
8,but,367,226,141,61.58
9,then,247,187,60,75.71


wrote the method diagnostic table and figure


## 11. Final checks

In [12]:
checks = [
    ("every run has 191 justifications",
     (run_level["n_justifications"] == 191).all()),
    ("stochastic and greedy kept separate",
     set(run_level["decoding_group"].astype(str)) == {"Stochastic", "Greedy"}),
    ("three stochastic runs, one greedy run, per model",
     f1.set_index(["model", "decoding_group"])["n_runs"].to_dict()
     == {(m, d): n for m in fin.MODEL_ORDER
         for d, n in (("Stochastic", 3), ("Greedy", 1))}),
    ("greedy SD is undefined",
     f1.loc[f1["decoding_group"].astype(str).eq("Greedy"),
            "relations_per_100_words_sd"].isna().all()),
    ("class counts sum to the accepted relations",
     int(run_level[[f"n_{c}" for c in fin.PDTB_TOP_LEVEL]].to_numpy().sum())
     == len(accepted)),
    ("composition rows sum to 100",
     np.allclose(
         f3[[f"{c}_pct_of_relations_mean" for c in fin.CATEGORY_ORDER]].sum(axis=1),
         100.0)),
    ("level-2 senses sum to the accepted relations",
     int(f4["total_count"].sum()) == len(accepted)),
    ("bootstrap covers 5 metrics x 3 pairs x 2 decodings",
     len(f5) == 30),
    ("no experimental relation entered the analysis",
     "provenance" not in accepted.columns),
]

for label, ok in checks:
    print(f"  [{'OK  ' if ok else 'FAIL'}] {label}")
assert all(ok for _, ok in checks), "a final check failed"
print("\nAll final checks passed.")

  [OK  ] every run has 191 justifications
  [OK  ] stochastic and greedy kept separate
  [OK  ] three stochastic runs, one greedy run, per model
  [OK  ] greedy SD is undefined
  [OK  ] class counts sum to the accepted relations
  [OK  ] composition rows sum to 100
  [OK  ] level-2 senses sum to the accepted relations
  [OK  ] bootstrap covers 5 metrics x 3 pairs x 2 decodings
  [OK  ] no experimental relation entered the analysis

All final checks passed.
